# 03 — Train (Single Fold)

**Parameterized by `FOLD_ID` (0–4). Launch 5 copies simultaneously, one per fold.**

How to run in parallel:
1. Open 5 separate Colab runtimes (requires Colab Pro/Pro+ for concurrent GPU runtimes)
2. In each runtime, set `FOLD_ID` in the parameter cell below to 0, 1, 2, 3, or 4
3. Run all cells — each runtime trains exactly one fold and saves its checkpoint to Drive
4. Once all 5 are done, run `04_infer.ipynb`

Alternatively, set `FOLD_ID = None` to run all 5 folds sequentially in a single runtime.

**Resumable:** If a runtime disconnects, just re-run. The notebook detects the
per-epoch in-progress checkpoint on Drive and picks up from the last completed epoch.

In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────────
!pip install -q transformers sentencepiece faiss-cpu

import os
try:
    import google.colab
    COLAB = True
except ImportError:
    COLAB = False

BASE_PATH = '/content/drive/MyDrive/Amazon/student_resource 2'  # EDIT

if COLAB and BASE_PATH.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = os.path.expanduser(BASE_PATH)
assert os.path.isdir(DATA_DIR), f'Folder not found at {DATA_DIR}'
print('Dataset OK at', DATA_DIR)

WORK_ROOT = '/content/er'
os.makedirs(WORK_ROOT, exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Colab ready.')

## ▶ PARAMETER CELL — set FOLD_ID before running

In [ ]:
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  Set FOLD_ID to 0, 1, 2, 3, or 4 for parallel execution.              │
# │  Set FOLD_ID = None to run all folds sequentially in one runtime.      │
# └─────────────────────────────────────────────────────────────────────────┘
FOLD_ID = None   # <-- CHANGE THIS

# Can also be set from an environment variable for scripted launches:
import os
_env_fold = os.environ.get('FOLD_ID', '')
if _env_fold.strip().isdigit():
    FOLD_ID = int(_env_fold)

if FOLD_ID is not None:
    assert 0 <= FOLD_ID <= 4, 'FOLD_ID must be 0–4'
    print(f'Running FOLD_ID = {FOLD_ID}')
else:
    print('FOLD_ID = None → will run all 5 folds sequentially')

## Section 0 — Configuration

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import json
import gc
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold

In [ ]:
try:
    COLAB
except NameError:
    COLAB = False; DATA_DIR = None; WORK_ROOT = None

if COLAB:
    DATASET_ROOT = Path(DATA_DIR)
    ROOT = Path(WORK_ROOT)
else:
    ROOT = Path.cwd()
    if not (ROOT / 'dataset').exists():
        ROOT = Path('..').resolve()
    DATASET_ROOT = ROOT

EMB      = DATASET_ROOT / 'output/embeddings'
CKPT_DIR = DATASET_ROOT / 'output/checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

WORK = ROOT / 'working'; WORK.mkdir(exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED                = 42
ENCODER_DIM         = 768
NAME_DIM            = ENCODER_DIM
ADDR_DIM            = ENCODER_DIM
TOWER_DIM           = 512
BATCH_SIZE          = 128
EPOCHS              = 150
WARMUP_EPOCHS       = 10
LR                  = 3e-4
N_FOLDS             = 5
EARLY_STOP_PATIENCE = 20

is_cuda_available = torch.cuda.is_available()
DEVICE = torch.device('cuda' if is_cuda_available else 'cpu')
print(f'Device: {DEVICE}')
np.random.seed(SEED); torch.manual_seed(SEED)

## Load Pair Tensors from Drive

In [ ]:
print('Loading pair tensors from Drive...')
all_name_inputs = np.load(EMB / 'all_name_inputs.npy')
all_addr_inputs = np.load(EMB / 'all_addr_inputs.npy')
all_scalars     = np.load(EMB / 'all_scalars.npy')
all_labels      = np.load(EMB / 'all_labels.npy')
all_groups      = np.load(EMB / 'all_groups.npy', allow_pickle=True)

print(f'Pairs loaded: {len(all_labels):,}  '
      f'(pos: {all_labels.sum():.0f}, neg: {(1-all_labels).sum():.0f})')

## Section 3D — Dataset Class

In [ ]:
class PairDataset(Dataset):
    def __init__(self, name_inputs, addr_inputs, scalars, labels):
        self.name_inputs = torch.from_numpy(name_inputs)
        self.addr_inputs = torch.from_numpy(addr_inputs)
        self.scalars     = torch.from_numpy(scalars)
        self.labels      = torch.from_numpy(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.name_inputs[idx],
            self.addr_inputs[idx],
            self.scalars[idx],
            self.labels[idx]
        )

## Section 3E — Tower + Adapter Model

In [ ]:
class ModalTowerAdapter(nn.Module):
    """
    Two-tower architecture with shared adapter head.

    Name Tower:  concat(s1_name_emb, cand_name_emb) → 512-dim
    Addr Tower:  concat(s1_addr_emb, cand_addr_emb) → 512-dim
    Adapter:     fuses tower outputs + 3 scalar features → match logit
    """
    def __init__(self, name_dim=NAME_DIM*2, addr_dim=ADDR_DIM*2,
                 tower_dim=TOWER_DIM, scalar_dim=3, dropout=0.1):
        super().__init__()

        def make_tower(in_dim):
            return nn.Sequential(
                nn.Linear(in_dim, tower_dim), nn.ReLU(), nn.LayerNorm(tower_dim), nn.Dropout(dropout),
                nn.Linear(tower_dim, tower_dim), nn.ReLU(), nn.LayerNorm(tower_dim), nn.Dropout(dropout),
                nn.Linear(tower_dim, tower_dim), nn.ReLU(), nn.LayerNorm(tower_dim), nn.Dropout(dropout),
            )

        self.name_tower = make_tower(name_dim)
        self.addr_tower = make_tower(addr_dim)

        fusion_dim = tower_dim + tower_dim + scalar_dim  # 1027
        self.adapter = nn.Sequential(
            nn.Linear(fusion_dim, 1024), nn.ReLU(), nn.LayerNorm(1024), nn.Dropout(dropout),
            nn.Linear(1024, 512),        nn.ReLU(), nn.LayerNorm(512),  nn.Dropout(dropout),
            nn.Linear(512, 256),         nn.ReLU(), nn.LayerNorm(256),  nn.Dropout(dropout),
            nn.Linear(256, 1)
            # No sigmoid — use BCEWithLogitsLoss during training
        )

    def forward(self, name_input, addr_input, scalars):
        name_out = self.name_tower(name_input)
        addr_out = self.addr_tower(addr_input)
        fused    = torch.cat([name_out, addr_out, scalars], dim=1)
        return self.adapter(fused).squeeze(1)

## Section 3F — Macro F0.5 Metric

In [ ]:
def macro_f05(labels, predictions, groups, all_s1_ids=None):
    """
    Compute macro-averaged F0.5.
    - labels:      binary numpy array
    - predictions: binary numpy array (after threshold)
    - groups:      numpy array of source1_entity_id per pair
    - all_s1_ids:  full list of S1 IDs (includes singletons with zero candidates)
    """
    by_source = defaultdict(list)
    for i, sid in enumerate(groups):
        by_source[sid].append(i)

    if all_s1_ids is not None:
        for sid in all_s1_ids:
            if sid not in by_source:
                by_source[sid] = []

    scores = []
    for sid, indices in by_source.items():
        if not indices:
            scores.append(1.0)
            continue
        t  = labels[indices].astype(bool)
        p  = predictions[indices].astype(bool)
        tp = np.sum(t & p)
        fp = np.sum(~t & p)
        fn = np.sum(t & ~p)
        prec = tp / max(tp + fp, 1)
        rec  = tp / max(tp + fn, 1)
        if prec + rec == 0:
            scores.append(1.0 if (not t.any() and not p.any()) else 0.0)
        else:
            scores.append(1.25 * prec * rec / (0.25 * prec + rec))

    return float(np.mean(scores)) if scores else 0.0

## Section 3G — Training Loop

In [ ]:
def train_fold(fold, model, train_loader, val_loader, val_groups_arr,
               epochs=EPOCHS, warmup=WARMUP_EPOCHS, patience=EARLY_STOP_PATIENCE):
    """
    Train one fold. Saves an in-progress checkpoint after every epoch to Drive
    (CKPT_DIR/fold_N_inprogress.pt) and resumes from it if interrupted.
    """
    ckpt_path = CKPT_DIR / f'fold_{fold}_inprogress.pt'

    pos_count = sum(batch[3].sum().item() for batch in train_loader)
    total     = len(train_loader.dataset)
    neg_count = total - pos_count
    pos_weight = torch.tensor([max(1.0, neg_count / max(pos_count, 1))]).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def lr_lambda(epoch):
        if epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / max(epochs - warmup, 1)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    start_epoch = 0
    best_score  = -1.0
    best_state  = None
    epochs_since_improve = 0

    # ── Resume from Drive checkpoint if interrupted ──────────────────────────
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        start_epoch          = ckpt['epoch'] + 1
        best_score           = ckpt['best_score']
        best_state           = ckpt['best_state']
        epochs_since_improve = ckpt.get('epochs_since_improve', 0)
        print(f'  Resuming fold {fold} from epoch {start_epoch} '
              f'(best F0.5 so far: {best_score:.4f})')

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0.0
        for name_inp, addr_inp, scalars, labels in train_loader:
            name_inp = name_inp.to(DEVICE)
            addr_inp = addr_inp.to(DEVICE)
            scalars  = scalars.to(DEVICE)
            labels   = labels.to(DEVICE)
            if len(labels) < 2:
                continue
            optimizer.zero_grad()
            logits = model(name_inp, addr_inp, scalars)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # Validation
        model.eval()
        val_probs = []
        with torch.no_grad():
            for name_inp, addr_inp, scalars, labels in val_loader:
                logits = model(
                    name_inp.to(DEVICE),
                    addr_inp.to(DEVICE),
                    scalars.to(DEVICE)
                )
                val_probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())

        val_probs_arr  = np.array(val_probs)
        val_labels_arr = val_loader.dataset.labels.numpy()

        best_thresh, best_val = 0.5, -1.0
        for t in np.linspace(0.30, 0.95, 66):
            score = macro_f05(val_labels_arr, val_probs_arr >= t, val_groups_arr)
            if score > best_val:
                best_val, best_thresh = score, t

        if best_val > best_score:
            best_score = best_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1}/{epochs}  loss {total_loss/len(train_loader):.4f}  '
                  f'val F0.5 {best_val:.4f}  thresh {best_thresh:.3f}  '
                  f'lr {scheduler.get_last_lr()[0]:.2e}')

        # ── Checkpoint every epoch (worst case: lose < 1 epoch on disconnect) ─
        torch.save({
            'epoch':              epoch,
            'model_state':        model.state_dict(),
            'optimizer_state':    optimizer.state_dict(),
            'scheduler_state':    scheduler.state_dict(),
            'best_score':         best_score,
            'best_state':         best_state,
            'epochs_since_improve': epochs_since_improve,
        }, ckpt_path)

        if epochs_since_improve >= patience:
            print(f'  Early stop fold {fold} at epoch {epoch+1} '
                  f'(no improvement in {patience} epochs)')
            break

    ckpt_path.unlink(missing_ok=True)  # clean up in-progress file
    return best_state, best_score

## Section 3H — Fold Training

In [ ]:
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_splits = list(kfold.split(all_name_inputs))  # [(train_idx, val_idx), ...]

# Determine which fold(s) to train
if FOLD_ID is not None:
    folds_to_run = [FOLD_ID]            # single-fold (parallel mode)
else:
    folds_to_run = list(range(N_FOLDS)) # all folds (sequential mode)

fold_results = {}  # fold_number → best_score

for fold_zero in folds_to_run:
    fold = fold_zero + 1  # 1-indexed for display and file naming
    train_idx, val_idx = all_splits[fold_zero]
    print(f'\n--- Fold {fold}/{N_FOLDS} ---')

    train_ds = PairDataset(
        all_name_inputs[train_idx], all_addr_inputs[train_idx],
        all_scalars[train_idx],     all_labels[train_idx]
    )
    val_ds = PairDataset(
        all_name_inputs[val_idx], all_addr_inputs[val_idx],
        all_scalars[val_idx],     all_labels[val_idx]
    )
    val_groups = all_groups[val_idx]

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = ModalTowerAdapter().to(DEVICE)

    final_path = CKPT_DIR / f'fold_{fold}_best.pt'
    if final_path.exists():
        print(f'  Found saved fold {fold} checkpoint on Drive — skipping training')
        saved = torch.load(final_path, map_location=DEVICE)
        best_score = saved['best_score']
    else:
        best_state, best_score = train_fold(fold, model, train_loader, val_loader, val_groups)
        torch.save({'best_state': best_state, 'best_score': best_score}, final_path)
        print(f'Fold {fold} checkpoint saved → {final_path.name}')

    fold_results[fold] = best_score
    print(f'Fold {fold} best val F0.5: {best_score:.4f}')

print('\n=== Fold results ===')
for f, s in fold_results.items():
    print(f'  Fold {f}: {s:.4f}')

## (Optional) Compute Full OOF Score

Only meaningful when running all 5 folds in the same runtime (`FOLD_ID = None`).
Skip this if running in parallel mode — run `04_infer.ipynb` instead, which
does a final threshold sweep using all checkpoints.

In [ ]:
if FOLD_ID is None:
    print('Computing OOF predictions across all folds...')
    oof_probs = np.zeros(len(all_labels), dtype=np.float32)

    for fold_zero in range(N_FOLDS):
        fold = fold_zero + 1
        train_idx, val_idx = all_splits[fold_zero]
        final_path = CKPT_DIR / f'fold_{fold}_best.pt'
        saved = torch.load(final_path, map_location=DEVICE)
        model = ModalTowerAdapter().to(DEVICE)
        model.load_state_dict(saved['best_state'])
        model.eval()

        val_ds = PairDataset(
            all_name_inputs[val_idx], all_addr_inputs[val_idx],
            all_scalars[val_idx],     all_labels[val_idx]
        )
        loader = DataLoader(val_ds, batch_size=512, shuffle=False)
        probs = []
        with torch.no_grad():
            for name_inp, addr_inp, scalars, _ in loader:
                logits = model(name_inp.to(DEVICE), addr_inp.to(DEVICE), scalars.to(DEVICE))
                probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        oof_probs[val_idx] = probs

    best_threshold, best_oof_score = 0.5, -1.0
    for t in np.linspace(0.30, 0.95, 66):
        score = macro_f05(all_labels, oof_probs >= t, all_groups)
        if score > best_oof_score:
            best_oof_score, best_threshold = score, float(t)

    print(f'\nFinal OOF macro-F0.5: {best_oof_score:.4f} at threshold {best_threshold:.3f}')
    (WORK / 'validation_metrics.json').write_text(json.dumps({
        'oof_macro_f05': best_oof_score,
        'threshold':     best_threshold,
        'folds':         N_FOLDS,
        'epochs':        EPOCHS
    }, indent=2))
    print(f'Saved validation_metrics.json')
else:
    print(f'Parallel mode (FOLD_ID={FOLD_ID}) — skipping OOF sweep.')
    print('Run 04_infer.ipynb once all 5 folds are done to get the full OOF score.')

## Done

Checkpoint(s) saved to `DATASET_ROOT/output/checkpoints/`:
```
fold_1_best.pt
fold_2_best.pt
fold_3_best.pt
fold_4_best.pt
fold_5_best.pt
```

Next step: once all 5 folds are complete, run `04_infer.ipynb`.